In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql.functions import col, initcap

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"
# Read circuits data from bronze layer
bronze_circuits_df = spark.table(bronze_table)

#print(f"Total records in bronze: {bronze_circuits_df.count()}") #82

In [0]:
# Apply transformations
silver_circuits_df = (
    bronze_circuits_df
    # Remove records with NULL circuit_id
    .filter(col("circuitid").isNotNull())
    # Remove duplicates
    .dropDuplicates(["circuitid"])
    # Drop URL column
    .drop("url")
    # Rename columns to snake_case and meaningful names using dictionary
    .withColumnsRenamed({
        "circuitid": "circuit_id",
        "circuitname": "circuit_name",
        "lat": "latitude",
        "long": "longitude"
    })
    # Transform circuit_name and locality to Title Case
    .withColumn("circuit_name", initcap(col("circuit_name")))
    .withColumn("locality", initcap(col("locality")))
)

# print(f"Total records after transformations: {silver_circuits_df.count()}")
# silver_circuits_df.display()

In [0]:
# Write transformed data to silver layer
(silver_circuits_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(silver_table))

print(f"✓ Circuits data successfully written to {catalog_name}.{silver_schema}.circuits")

In [0]:
# spark.table(silver_table).limit(10).display()